In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_care_site AS 
SELECT
NULL AS care_site_name,
NULL AS place_of_service_concept_id,
NULL AS location_id,
CONCAT('allscripts_tw', CHAR(31), 'dbo_billing_location_de', CHAR(31), 'id', CHAR(31), CAST(dbo_billing_location_de.id AS BIGINT)) AS care_site_source_value,
NULL AS place_of_service_source_value,
'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_billing_location_de
WHERE 1=1  
AND dbo_billing_location_de.id IS NOT NULL
AND dbo_billing_location_de.isinactiveflag = 'N'
UNION ALL
SELECT
NULL AS care_site_name,
NULL AS place_of_service_concept_id,
NULL AS location_id,
CONCAT('allscripts_tw', CHAR(31), 'dbo_location_de', CHAR(31), 'id', CHAR(31), CAST(dbo_location_de.id AS BIGINT)) AS care_site_source_value,
NULL AS place_of_service_source_value,
'allscripts_tw' AS source_system
FROM  _exponent._bronze_allscripts_tw_works_vw.dbo_location_de
WHERE 1=1  
AND dbo_location_de.id IS NOT NULL
AND dbo_location_de.isinactiveflag = 'N'
UNION ALL
SELECT
NULL AS care_site_name,
NULL AS place_of_service_concept_id,
NULL AS location_id,
CONCAT('allscripts_tw', CHAR(31), 'dbo_site_de', CHAR(31), 'id', CHAR(31), CAST(dbo_site_de.id AS BIGINT)) AS care_site_source_value,
NULL AS place_of_service_source_value,
'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_site_de
WHERE 1=1  
AND dbo_site_de.id IS NOT NULL
AND dbo_site_de.isinactiveflag = 'N'
UNION ALL
SELECT
NULL AS care_site_name,
NULL AS place_of_service_concept_id,
NULL AS location_id,
CONCAT('allscripts_tw', CHAR(31), 'dbo_site_location_de', CHAR(31), 'id', CHAR(31), CAST(dbo_site_location_de.id AS BIGINT)) AS care_site_source_value,
NULL AS place_of_service_source_value,
'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_site_location_de
WHERE 1=1  
AND dbo_site_location_de.id IS NOT NULL
AND dbo_site_location_de.isinactiveflag = 'N'


In [0]:
%sql
MERGE INTO _exponent.omop_silver.care_site AS target
USING silver_care_site AS source
ON target.care_site_source_value = source.care_site_source_value

WHEN MATCHED AND NOT (
     target.care_site_name                <=> source.care_site_name
 AND target.place_of_service_concept_id   <=> source.place_of_service_concept_id
 AND target.location_id                   <=> source.location_id
 AND target.place_of_service_source_value <=> source.place_of_service_source_value
 AND target.source_system                 <=> source.source_system
) THEN UPDATE SET
  target.care_site_name                = source.care_site_name,
  target.place_of_service_concept_id   = source.place_of_service_concept_id,
  target.location_id                   = source.location_id,
  target.care_site_source_value        = source.care_site_source_value,
  target.place_of_service_source_value = source.place_of_service_source_value,
  target.source_system                 = source.source_system,
  target.last_mod_tsp                  = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  care_site_name,
  place_of_service_concept_id,
  location_id,
  care_site_source_value,
  place_of_service_source_value,
  source_system,
  last_mod_tsp
) VALUES (
  source.care_site_name,
  source.place_of_service_concept_id,
  source.location_id,
  source.care_site_source_value,
  source.place_of_service_source_value,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_care_site (
    source_system,
    care_site_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT DISTINCT
    care_site.source_system,
    care_site.care_site_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(care_site.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM _exponent.omop_silver.care_site care_site
LEFT ANTI JOIN _exponent.omop_mapping.source_to_care_site source_to_care_site
  ON care_site.source_system = source_to_care_site.source_system
 AND care_site.care_site_source_value = source_to_care_site.care_site_source_value
WHERE care_site.care_site_source_value IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold AS
SELECT
  source_to_care_site.care_site_id,
  care_site.care_site_name,
  care_site.place_of_service_concept_id,
  care_site.location_id,
  care_site.care_site_source_value,
  care_site.place_of_service_source_value

FROM _exponent.omop_silver.care_site

JOIN _exponent.omop_mapping.source_to_care_site
  ON care_site.care_site_source_value =
     source_to_care_site.care_site_source_value
 AND care_site.source_system =
     source_to_care_site.source_system
 AND source_to_care_site.active_flag = TRUE;

In [0]:
%sql
MERGE INTO _exponent.omop.care_site AS target
USING gold AS source
ON target.care_site_id = source.care_site_id

WHEN MATCHED AND NOT (
     target.care_site_name                <=> source.care_site_name
 AND target.place_of_service_concept_id   <=> source.place_of_service_concept_id
 AND target.location_id                   <=> source.location_id
 AND target.care_site_source_value        <=> source.care_site_source_value
 AND target.place_of_service_source_value <=> source.place_of_service_source_value
) THEN UPDATE SET
  target.care_site_name                = source.care_site_name,
  target.place_of_service_concept_id   = source.place_of_service_concept_id,
  target.location_id                   = source.location_id,
  target.care_site_source_value        = source.care_site_source_value,
  target.place_of_service_source_value = source.place_of_service_source_value

WHEN NOT MATCHED THEN INSERT (
  care_site_id,
  care_site_name,
  place_of_service_concept_id,
  location_id,
  care_site_source_value,
  place_of_service_source_value
) VALUES (
  source.care_site_id,
  source.care_site_name,
  source.place_of_service_concept_id,
  source.location_id,
  source.care_site_source_value,
  source.place_of_service_source_value
);